In [1]:
from __future__ import division
import pandas as pd 
import numpy as np 
import datetime
import os
from pyomo.environ import *
from pyomo.opt import SolverFactory
from pyomo.gdp import *
import pyutilib.subprocess.GlobalData

In [2]:
import pyodbc

In [3]:
import timeit

In [4]:
import sys
sys.path.append('..')
from autoplan_week import WeekAutoPlanner
from test_odbc_style_query import all_menu_opts_qry_odbc, query_datasets

In [5]:
import json
with open("../test/autoplan_week_test_params1.json") as file:
    params = json.load(file)

In [6]:
small_solve_params = {
    'requirements_dict':{
        'day1':params['params']['requirements_dict']['day1']
    },
    'menus_dict_list':[params['params']['menus_dict_list'][0]],
    'week_start_dt':params['params']['week_start_dt'],
    'week_end_dt':params['params']['week_end_dt']
}

In [7]:
small_solve_params

{'requirements_dict': {'day1': {'calories': [1995, 2205],
   'protein_g': [175, 193],
   'fat_g': [89, 98],
   'carb_g': [125, 138]}},
 'menus_dict_list': [{'day': 'day1',
   'meals': [{'meal': 'Breakfast', 'menus': [{'type': 'menu', 'id': 5}]},
    {'meal': 'Lunch', 'menus': [{'type': 'menu', 'id': 6}]},
    {'meal': 'Dinner', 'menus': [{'type': 'menu', 'id': 7}]},
    {'meal': 'Snack', 'menus': [{'type': 'menu', 'id': 8}]}]}],
 'week_start_dt': '2019-11-17',
 'week_end_dt': '2019-11-23'}

In [8]:
med_solve_params = {
    'requirements_dict':{
        'day1':params['params']['requirements_dict']['day1'],
        'day2':params['params']['requirements_dict']['day2'],
        'day3':params['params']['requirements_dict']['day3'],
    },
    'menus_dict_list':params['params']['menus_dict_list'][0:3],
    'week_start_dt':params['params']['week_start_dt'],
    'week_end_dt':params['params']['week_end_dt']
}

In [9]:
large_solve_params = {
    'requirements_dict':{
        'day1':params['params']['requirements_dict']['day1'],
        'day2':params['params']['requirements_dict']['day2'],
        'day3':params['params']['requirements_dict']['day3'],
        'day4':params['params']['requirements_dict']['day4'],
        'day5':params['params']['requirements_dict']['day5'],
        'day6':params['params']['requirements_dict']['day6'],
    },
    'menus_dict_list':params['params']['menus_dict_list'],
    'week_start_dt':params['params']['week_start_dt'],
    'week_end_dt':params['params']['week_end_dt']
}

In [10]:
test_params = large_solve_params

In [11]:
# Params
connection = pyodbc.connect('DSN=PostgreSQL30;PWD=v2nc3123')

In [12]:
# WeekPLannerMod
class WeekAutoPlannerMod(WeekAutoPlanner):
    
    def solve_weekplan_model_dev(self,solver_rel_path, model):
        """
        Solves and returns model
        """
        goal_dir = os.path.join(os.getcwd(),solver_rel_path)
        solverpath_exe = os.path.normpath(goal_dir)
        #opt = SolverFactory('gdpbb',executable=solverpath_exe) # Can change here with newest version
        opt = SolverFactory('cbc') # Can change here with newest version
        #opt.options["threads"] = 4
        opt.options["maxSolutions"] = 2
        #opt.options["maxNodes"] = 1
        #opt.options["blah"] = 1
        #opt.options["maxSeconds"] = 2
        opt.solve(model)
        return model, opt
    

In [13]:
planner = WeekAutoPlannerMod(
            connection=connection,
            user_id=2,
            requirements_dict=test_params['requirements_dict'], 
            menus_dict_list=test_params['menus_dict_list'],
            week_start_dt=test_params['week_start_dt'], 
            week_end_dt=test_params['week_end_dt'], 
            n_snack=5, 
            solver_rel_path='../../../plm_env/lib/python3.6/site-packages/Cbc-2.9.8',
            server_type='dev'
    )
food_df, meals_idx_tup, days_list = query_datasets(connection=connection,
                                                   user_id=planner.user_id,
                                                   menus_list=planner.menus_list, 
                                                   week_start_dt=planner.week_start_dt, 
                                                   week_end_dt=planner.week_end_dt)
opt_input_df, leftover_dict =planner.build_opt_dataset(planner.menus_list, food_df)
food_max, global_m, meal_food_tup, src_constr_dict, incl_dict = planner.set_opt_variables(opt_input_df, days_list, meals_idx_tup)
model = planner.build_autoplan_week_model(opt_input_df,planner.requirements_dict,food_max,5,global_m,
    src_constr_dict,incl_dict,planner.sides_dict,meal_food_tup, meals_idx_tup, days_list, leftover_dict)

(0,5,6,7,8,6,7,8,5,6,7,8,5,6,7,8,6,7,8,5,6,7,8)
(0)
(0,368,368)


..\autoplan_week.py:357: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  loop_df['meal'] = meal['meal']


In [14]:
start = timeit.default_timer()
solved_model, opt = planner.solve_weekplan_model_dev(planner.solver_rel_path+'/bin/cbc', model)
end = timeit.default_timer()
total_time = end-start
print(total_time)

***WARNING: CBC plugin currently not processing solution status=Stopped correctly. Full status line is: Stopped on iterations - objective value 20.88254029
77.46089692564465


In [104]:
opt

In [105]:
planner.parse_weekplan_model_results(solved_model, opt_input_df, leftover_dict)

,unique_id,day,meal,dish_num,fd_type,amt,serving_size_val
8,303290,day1,Breakfast,Sides,food,1.0,1.00
15,302396,day1,Breakfast,Main Courses,food,1.0,1.00
28,469840,day1,Lunch,Whole Meals,food,2.0,1.00
61,473073,day1,Dinner,Sides,food,1.0,1.00
67,469840,day1,Dinner,Main Courses,food,2.0,1.00
78,329244,day1,Snack,Snacks,food,1.0,1.00
101,282958,day2,Breakfast,Whole Meal,food,1.0,1.00
103,429798,day2,Breakfast,Whole Meal,food,1.0,2.00
104,429838,day2,Breakfast,Whole Meal,food,2.0,1.00
115,54293,day2,Lunch,Sides,food,1.0,1.00
